# AI-854 Lecture 5: Can a correct visualization still mislead?

**Medical evidence: the mathematics behind the chart.** HPS, AIM-HIGH, and the NHANES linked-mortality study provide authentic aggregate evidence. Rows represent study arms or published estimates, never invented patients.

The instructor runs the code. Students predict, calculate on paper, explain and defend. Keep this notebook beside `data/`. Run cells in order. RUN 0–17 form the core route. RUN 18 is an optional area-encoding extension.

The lecture lasts 180 minutes with a break at 105–120. See `study_dictionary.md` for study scope. The executed notebook is the recovery copy.

**Model boundary:** introductory Bernoulli approximations use independent arms and observed counts. These calculations do not reproduce the original survival models or refit adjusted sugar estimates. No empirical survival curves can be reconstructed from these tables alone.

In [ ]:
# Load the published lecture inputs in Colab; verify local inputs offline.
from pathlib import Path
import hashlib, sys
from urllib.request import urlopen
PUBLIC_ROOT = Path.cwd()
_files = {'data/hdl_summary.csv': 'e34dd4fb7f91c43cf73873675e593783b897604a70b52f741e174e3019b0ef44', 'data/reported_estimates.csv': 'f616fdc3697971f196e12ff3f09f97d94165acdf7b9b44ae29cb502596951922', 'data/sugar_estimates.csv': 'c965f36aeceff45f5a90b44a339ee5903e25df408c75a94774e80c0329750171', 'data/trial_events.csv': '678056c3393219be3d387ea16fb0d218a433269d307eaee2a012b25be72778b9'}
_base_url = 'https://raw.githubusercontent.com/DrFarrukh/teaching/main/courses/data-analysis-and-visualization/lectures/lecture-05/'
_in_colab = 'google.colab' in sys.modules
for _name, _sha in _files.items():
    _path = PUBLIC_ROOT / _name
    if not _path.exists():
        if not _in_colab:
            raise FileNotFoundError('Extract the student ZIP and run this notebook from its lecture folder. Missing: ' + _name)
        with urlopen(_base_url + _name, timeout=30) as _response:
            _bytes = _response.read()
        if hashlib.sha256(_bytes).hexdigest() != _sha:
            raise ValueError('Downloaded input failed its checksum: ' + _name)
        _path.parent.mkdir(parents=True, exist_ok=True)
        _path.write_bytes(_bytes)
    if hashlib.sha256(_path.read_bytes()).hexdigest() != _sha:
        raise ValueError('Input differs from the published snapshot: ' + _name)
print('Lecture inputs ready; frozen source checksums verified.')


## RUN 0: Load the evidence

**Predict:** what is one observation in each table? Identify the units before calculating.

In [ ]:
from pathlib import Path
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
from IPython.display import display

BASE = PUBLIC_ROOT
OUT = Path(os.environ.get('DATA_L5_OUTPUT_DIR', str(BASE / 'outputs')))
FIG = OUT / 'figures'
FIG.mkdir(parents=True, exist_ok=True)
trials = pd.read_csv(BASE / 'data/trial_events.csv')
hdl = pd.read_csv(BASE / 'data/hdl_summary.csv')
sugar = pd.read_csv(BASE / 'data/sugar_estimates.csv')
reported = pd.read_csv(BASE / 'data/reported_estimates.csv')
COLORS = ['#167d9a', '#d46a37']
plt.rcParams.update({'font.size': 13, 'axes.spines.top': False,
    'axes.spines.right': False, 'figure.dpi': 120, 'savefig.dpi': 180,
    'svg.fonttype': 'none'})
def save(fig, name):
    fig.tight_layout()
    fig.savefig(FIG / f'{name}.png', bbox_inches='tight')
    fig.savefig(FIG / f'{name}.svg', bbox_inches='tight')
    plt.show()
display(trials[['study', 'arm', 'n', 'patients_with_event', 'outcome']])
display(sugar[['quintile', 'midvalue_pct', 'model', 'hr', 'ci_low', 'ci_high']])

## RUN 1: Audit the extracted values

**Predict:** should Q1 have a confidence interval of [1, 1]? A reference is a defined comparison, not an estimated effect with zero uncertainty.

In [ ]:
assert not trials.duplicated(['study', 'arm']).any()
assert (trials.n > 0).all()
assert np.equal(trials.n, np.floor(trials.n)).all()
assert np.equal(trials.patients_with_event, np.floor(trials.patients_with_event)).all()
assert trials.patients_with_event.between(0, trials.n).all()
assert not sugar.duplicated(['quintile', 'model']).any()
assert not hdl.duplicated(['study', 'arm', 'months']).any()
assert (hdl.hdl_mg_dl > 0).all()
for model, table in sugar.groupby('model'):
    assert table.quintile.tolist() == ['Q1', 'Q2', 'Q3', 'Q4', 'Q5']
    assert table.midvalue_pct.is_monotonic_increasing
    assert table.is_reference.sum() == 1
ref = sugar.loc[sugar.is_reference]
assert (ref.hr == 1).all() and ref[['ci_low', 'ci_high']].isna().all().all()
nonref = sugar.loc[~sugar.is_reference]
assert (nonref.ci_low > 0).all()
assert (nonref.ci_low <= nonref.hr).all() and (nonref.hr <= nonref.ci_high).all()
assert (reported.ci_low <= reported.estimate).all()
assert (reported.estimate <= reported.ci_high).all()
for table in [trials, hdl, sugar, reported]:
    assert table.source.str.startswith('https://').all()
print('Audit passed: counts, keys, intervals, reference rows, units and source links.')

## RUN 2: HPS risks and effect sizes

**Paper first:** calculate 2033/10269 and 2585/10267. Will 5.38 percentage points and 21.4% describe the same quantity?

In [ ]:
def trial_effect(table):
    T = table.loc[table.role == 'treatment'].iloc[0]
    C = table.loc[table.role == 'control'].iloc[0]
    pt, pc = T.patients_with_event / T.n, C.patients_with_event / C.n
    arr = pc - pt
    rr = pt / pc
    se_arr = np.sqrt(pc * (1-pc) / C.n + pt * (1-pt) / T.n)
    # The log-RR approximation requires positive event counts.
    if min(T.patients_with_event, C.patients_with_event) <= 0:
        raise ValueError('The log-RR formula requires nonzero event counts.')
    se_log = np.sqrt(1/T.patients_with_event - 1/T.n
                     + 1/C.patients_with_event - 1/C.n)
    low, high = np.exp(np.log(rr) + np.array([-1, 1]) * 1.96 * se_log)
    return dict(p_t=pt, p_c=pc, arr=arr, rr=rr, rrr=1-rr,
        se_arr=se_arr, arr_low=arr-1.96*se_arr, arr_high=arr+1.96*se_arr,
        se_log_rr=se_log, rr_low=low, rr_high=high)
hps = trials.loc[trials.study == 'HPS'].copy()
H = trial_effect(hps)
display(pd.Series(H).to_frame('Count-based estimate'))
print(f'HPS: {100*H["p_t"]:.2f}% vs {100*H["p_c"]:.2f}%')
print(f'ARR = {100*H["arr"]:.2f} percentage points; RRR = {100*H["rrr"]:.2f}%')

## RUN 3: Uncertainty in absolute benefit

**Predict:** which null belongs on a difference plot? Derive the standard error on the board before running.

In [ ]:
print(f'Approximate 95% ARR interval: {100*H["arr_low"]:.2f} to {100*H["arr_high"]:.2f} percentage points')
print(f'Affected people per 1000: {1000*H["p_t"]:.1f} vs {1000*H["p_c"]:.1f}')
print(f'Estimated fewer affected people: {1000*H["arr"]:.1f}, 95% interval {1000*H["arr_low"]:.1f} to {1000*H["arr_high"]:.1f}')
fig, ax = plt.subplots(figsize=(10, 3.1))
ax.errorbar([H['arr']*1000], [0], xerr=[[1000*(H['arr']-H['arr_low'])],
    [1000*(H['arr_high']-H['arr'])]], fmt='o', capsize=7, color=COLORS[0])
ax.axvline(0, color='#637482', linestyle='--')
ax.set(yticks=[], xlim=(-5, 75), xlabel='Fewer participants with a major vascular event per 1,000',
       title='HPS absolute benefit during approximately five years')
ax.text(54, 0.08, '53.8 (approximate 95% CI 42.4–65.2)', ha='center')
save(fig, '01_hps_absolute_benefit')

## RUN 4: A baseline changes the apparent ratio

**Predict:** which group looks more than twice as large? This figure is deliberately incomplete.

In [ ]:
rates = np.array([H['p_t'], H['p_c']])*100
b = 15.0
apparent = (rates[1]-b)/(rates[0]-b)
actual = rates[1]/rates[0]
print(f'Apparent bar-height ratio at baseline {b}%: {apparent:.3f}')
print(f'Actual control/treatment risk ratio: {actual:.3f}')
fig, ax = plt.subplots(figsize=(10, 4.2))
ax.bar(['Simvastatin', 'Placebo'], rates, color=COLORS)
ax.set(ylim=(15, 28), ylabel='Participants with event (%)',
    title='Deliberately incomplete pitch: bar baseline 15%')
for i, v in enumerate(rates): ax.text(i, v+.4, f'{v:.2f}%', ha='center')
save(fig, '02_hps_truncated_bars')

## RUN 5: Repair the HPS display

**Explain:** what does a zero baseline repair, and what does it leave for the caption?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].bar(['Simvastatin', 'Placebo'], rates, color=COLORS)
axes[0].set(ylim=(0, 30), ylabel='Participants with event (%)', title='Length encoding: zero baseline')
for i, v in enumerate(rates): axes[0].text(i, v+.5, f'{v:.2f}%', ha='center')
axes[1].scatter(rates, [1, 0], c=COLORS, s=90)
axes[1].set(xlim=(18, 27), yticks=[0,1], yticklabels=['Placebo','Simvastatin'],
            xlabel='Participants with event (%)', title='Position encoding: restricted axis')
save(fig, '03_hps_repaired_encodings')

## RUN 6: Equal denominators reveal absolute size

**Predict:** about how many of 1,000 comparable participants experience the endpoint in each arm?

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 5))
for ax, rate, label, color in zip(axes, [H['p_t'],H['p_c']], ['Simvastatin','Placebo'], COLORS):
    count = round(1000*rate)
    k = np.arange(1000)
    ax.scatter(k%50, -(k//50), c=np.where(k<count, color, '#dce3e6'), s=10)
    ax.set_title(f'{label}: approximately {count} affected per 1,000')
    ax.set_axis_off()
fig.suptitle('Scaled illustration of observed risks, not reconstructed patient records', fontsize=13)
save(fig, '04_hps_equal_denominators')

## RUN 7: Count ratios and reported survival analysis

**Explain:** why is the paper’s 24% reduction different from our 21.4%? Which effect did randomization identify?

In [ ]:
hps_report = reported.loc[reported.study == 'HPS'].iloc[0]
display(pd.DataFrame([
    {'Analysis':'Count-based cumulative RR', 'Estimate':H['rr'], 'Lower':H['rr_low'], 'Upper':H['rr_high']},
    {'Analysis':hps_report.estimand, 'Estimate':hps_report.estimate, 'Lower':hps_report.ci_low, 'Upper':hps_report.ci_high}
]))
print('Classroom result: assignment to simvastatin reduced the selected composite outcome.')
print('Wider LDL causality rests on converging evidence, not this treatment contrast alone.')

## RUN 8: The incomplete HDL exhibit

**Predict:** does a 20% rise in median HDL establish a 20% reduction in cardiovascular events?

In [ ]:
hdl_change = 100*(hdl.hdl_mg_dl.iloc[1]/hdl.hdl_mg_dl.iloc[0]-1)
print(f'Within-arm change in median HDL: {hdl_change:.1f}%')
fig, ax = plt.subplots(figsize=(10, 4.2))
ax.bar(hdl.time_point, hdl.hdl_mg_dl, color=COLORS[0])
ax.set(ylim=(0, 50), ylabel='Median HDL-C (mg/dL)',
       title='Incomplete exhibit: niacin arm only')
for i, v in enumerate(hdl.hdl_mg_dl): ax.text(i,v+1,str(v),ha='center')
save(fig, '05_hdl_incomplete_exhibit')

## RUN 9: Patient outcomes in AIM-HIGH

**Paper first:** compare the two event risks. Explain what the biomarker figure omitted.

In [ ]:
aim = trials.loc[trials.study == 'AIM-HIGH'].copy()
A = trial_effect(aim)
display(aim[['arm', 'n', 'patients_with_event']])
print(f'Niacin arm event risk {100*A["p_t"]:.2f}%; control {100*A["p_c"]:.2f}%')
fig, axes = plt.subplots(1,2,figsize=(12,4.5))
axes[0].bar(hdl.time_point, hdl.hdl_mg_dl, color=COLORS[0])
axes[0].set(ylim=(0,50),ylabel='Median HDL-C (mg/dL)',title='Niacin arm biomarker summary')
axes[1].bar(['Niacin + statin','Placebo + statin'], [100*A['p_t'],100*A['p_c']], color=COLORS)
axes[1].set(ylim=(0,22),ylabel='Participants with primary event (%)',title='Randomized outcome comparison')
save(fig, '06_hdl_and_outcomes')

## RUN 10: A log-risk-ratio confidence interval

**Derive first:** show why Var(log p-hat) is approximately (1−p)/(np). Then read the interval against 1.

In [ ]:
aim_report = reported.loc[reported.study == 'AIM-HIGH'].iloc[0]
print(f'Count-based AIM-HIGH RR {A["rr"]:.3f}, approximate 95% CI {A["rr_low"]:.3f}–{A["rr_high"]:.3f}')
print(f'Reported HR {aim_report.estimate:.2f}, 95% CI {aim_report.ci_low:.2f}–{aim_report.ci_high:.2f}')
fig, ax = plt.subplots(figsize=(10,4))
vals = [A['rr'], aim_report.estimate]
lo = [A['rr_low'], aim_report.ci_low]
hi = [A['rr_high'], aim_report.ci_high]
for y,v,l,u,c in zip([1,0], vals,lo,hi,COLORS):
    ax.errorbar(v,y,xerr=[[v-l],[u-v]],fmt='o',capsize=6,color=c)
ax.axvline(1,ls='--',c='#637482')
ax.set(xlim=(.8,1.3),yticks=[0,1],yticklabels=['Reported hazard ratio','Count-based risk ratio'],
       xlabel='Niacin arm / placebo arm',title='Two estimators with distinct definitions')
save(fig, '07_aim_ratio_intervals')

## RUN 11: Linear and logarithmic ratio geometry

**Predict:** should 0.5 and 2 sit equally far from 1? What about a difference axis?

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,4))
for ax in axes:
    ax.scatter([.5,1,2],[0,0,0],s=85,c=[COLORS[0],'#637482',COLORS[1]])
    ax.axvline(1,ls='--',c='#637482')
    ax.set(yticks=[],xlim=(.4,2.5),xlabel='Ratio (dimensionless)')
axes[0].set_title('Linear distance encodes addition')
axes[1].set_xscale('log')
axes[1].set_xticks([.5,1,2]); axes[1].xaxis.set_major_formatter(ScalarFormatter())
axes[1].minorticks_off(); axes[1].set_title('Log distance encodes multiplication')
save(fig, '08_ratio_geometry')
print('log(2) =', np.log(2), '; log(1/2) =', np.log(.5))

## RUN 12: Normalize added-sugar exposure

**Paper first:** 50 g at 2,000 kcal and 100 g at 4,000 kcal. These are invented arithmetic examples, not study observations.

In [ ]:
examples = pd.DataFrame({'added_sugar_g':[50,100], 'total_energy_kcal':[2000,4000]})
examples['added_sugar_energy_pct'] = 100*4*examples.added_sugar_g/examples.total_energy_kcal
display(examples)
print('The study table reports modeled usual exposure midvalues. A single day has within-person variation.')

## RUN 13: Selected endpoints omit the shape and precision

**Predict:** what is missing when a pitch shows only Q1 and Q5?

In [ ]:
S = sugar.loc[sugar.model == 'full'].copy()
selected = S.loc[S.quintile.isin(['Q1','Q5'])]
fig, ax = plt.subplots(figsize=(10,4.2))
ax.bar(selected.quintile, selected.hr, color=[COLORS[0],COLORS[1]])
ax.set(ylim=(0,3.6),ylabel='Reported adjusted hazard ratio',
       title='Deliberately incomplete: two groups, no intervals')
for i, v in enumerate(selected.hr): ax.text(i,v+.08,f'{v:.2f}',ha='center')
save(fig, '09_sugar_selected_endpoints')

## RUN 14: All five exposure midvalues with intervals

**Explain:** does 25.2% mark a category threshold? Does HR 2.03 tell us the absolute death probability?

In [ ]:
def plot_sugar(ax, table, color=COLORS[0], marker='o'):
    R = table.loc[table.is_reference]
    D = table.loc[~table.is_reference]
    ax.errorbar(D.midvalue_pct,D.hr,yerr=[D.hr-D.ci_low,D.ci_high-D.hr],
        fmt=marker,linestyle='none',capsize=5,color=color)
    ax.scatter(R.midvalue_pct,R.hr,marker=marker,c=color,s=55,facecolors='none')
    ax.axhline(1,ls='--',color='#637482')
    ax.set_yscale('log')
    ax.set_yticks([1,1.5,2,3,4]); ax.yaxis.set_major_formatter(ScalarFormatter())
    ax.minorticks_off()
    ax.set(xlim=(5,28),ylim=(.85,4),xlabel='Usual added sugar (% of total energy), quintile midvalue',
           ylabel='Reported adjusted HR for CVD mortality')
fig, ax = plt.subplots(figsize=(11,5))
plot_sugar(ax,S)
for _,row in S.iterrows(): ax.annotate(row.quintile,(row.midvalue_pct,row.hr),xytext=(8,5),textcoords='offset points')
ax.set_title('Full adjustment: five summaries, no invented dose-response curve')
ax.text(7.4,.90,'Q1 reference',ha='center',fontsize=10)
save(fig, '10_sugar_full_evidence')

## RUN 15: Compare both published adjustment models

**Predict:** can a lower estimate tell us which added covariate caused the change?

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(13,5),sharex=True,sharey=True)
for ax,model,color,title in zip(axes,['demographic','full'],COLORS,
    ['Age, sex, race/ethnicity','Full published adjustment']):
    plot_sugar(ax,sugar.loc[sugar.model == model],color)
    ax.set_title(title)
axes[1].set_ylabel('')
save(fig, '11_sugar_model_comparison')
print('Both models are adjusted. Full adjustment includes total serum cholesterol.')
print('The estimates do not identify a sugar-versus-LDL replacement effect.')

## RUN 16: Final evidence figure

**Defend:** why do these panels require separate endpoints, study designs and scales?

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(17,5.2))
axes[0].errorbar(H['arr']*1000,0,xerr=[[1000*(H['arr']-H['arr_low'])],
    [1000*(H['arr_high']-H['arr'])]],fmt='o',capsize=6,color=COLORS[0])
axes[0].axvline(0,ls='--',c='#637482')
axes[0].set(xlim=(-5,75),yticks=[],xlabel='Fewer affected per 1,000',title='HPS randomized treatment\nMajor vascular event, ~5 years')
axes[1].errorbar(aim_report.estimate,0,xerr=[[aim_report.estimate-aim_report.ci_low],
    [aim_report.ci_high-aim_report.estimate]],fmt='o',capsize=6,color=COLORS[0])
axes[1].axvline(1,ls='--',c='#637482'); axes[1].set_xscale('log')
axes[1].set(xlim=(.8,1.3),xticks=[.8,1,1.2],yticks=[],xlabel='Reported HR, niacin / placebo',
    title='AIM-HIGH randomized add-on\nPrimary composite outcome, ~3 years')
axes[1].xaxis.set_major_formatter(ScalarFormatter()); axes[1].minorticks_off()
plot_sugar(axes[2],S)
axes[2].set_title('NHANES observational cohort\nCVD mortality, full adjustment')
axes[2].set_xlabel('Usual added sugar (% of energy)')
axes[2].set_ylabel('Reported HR')
save(fig,'12_final_evidence')

## RUN 17: Export the calculations and conclusions

**Write:** one result, numerical support and scope for each pitch.

In [ ]:
results = pd.DataFrame([dict(study='HPS',**H),dict(study='AIM-HIGH',**A)])
results.to_csv(OUT / 'trial_calculations.csv',index=False)
sugar.to_csv(OUT / 'sugar_plot_values.csv',index=False)
payload = {'HPS':H,'AIM-HIGH':A,'hdl_percent_change':hdl_change,
    'hps_baseline_pct':b,'hps_apparent_height_ratio':apparent,
    'hps_control_to_treatment_risk_ratio':actual}
(OUT / 'results.json').write_text(json.dumps(payload,indent=2))
print('HPS: assignment to simvastatin reduced major vascular events.')
print('AIM-HIGH: the niacin add-on did not demonstrate additional clinical benefit in this setting.')
print('NHANES: higher modeled usual added-sugar intake was associated with higher CVD mortality after full adjustment.')
print('Multiple contributors remain compatible with the evidence.')

## RUN 18: Optional extension: area encoding

**Predict:** if a value doubles, should the radius double? This geometry exercise uses invented values.

In [ ]:
values = np.array([1.,2.])
print('Values:',values,'; correct radii:',np.sqrt(values),'; incorrect areas with radius=value:',values**2)
fig, axes = plt.subplots(1,2,figsize=(10,3))
for ax,sizes,title in zip(axes,[values*1600,values**2*1600],
                         ['Area proportional to value','Incorrect: radius proportional to value']):
    ax.scatter([0,1],[0,0],s=sizes,c=COLORS,alpha=.8)
    ax.set(xlim=(-.8,1.8),ylim=(-1,1),xticks=[0,1],xticklabels=['1 unit','2 units'],yticks=[],title=title)
save(fig,'13_area_encoding_extension')